# Microsoft Foundry · Agente com servidor MCP (Microsoft Learn)

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 17): integração de um agente com um **servidor MCP** (*Model Context Protocol*) já existente, o do **Microsoft Learn**, que dá acesso a toda a documentação, exemplos de código e trilhas de estudo da Microsoft.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Pré-requisito: registrar o servidor MCP no portal](#1.-Pré-requisito:-registrar-o-servidor-MCP-no-portal)
2. [Instalação das bibliotecas](#2.-Instalação-das-bibliotecas)
3. [Variáveis de ambiente](#3.-Variáveis-de-ambiente)
4. [Conectando ao projeto do Foundry](#4.-Conectando-ao-projeto-do-Foundry)
5. [Cliente OpenAI do projeto](#5.-Cliente-OpenAI-do-projeto)
6. [Obtendo o ID da conexão MCP](#6.-Obtendo-o-ID-da-conexão-MCP)
7. [Montando a ferramenta MCP](#7.-Montando-a-ferramenta-MCP)
8. [Criando o agente MCP](#8.-Criando-o-agente-MCP)
9. [Criando a conversa](#9.-Criando-a-conversa)
10. [Perguntando ao agente](#10.-Perguntando-ao-agente)
11. [Verificando no portal do Foundry](#11.-Verificando-no-portal-do-Foundry)
12. [Resumo dos aprendizados](#12.-Resumo-dos-aprendizados)
13. [Limpeza e custos](#13.-Limpeza-e-custos)
14. [Interface com Streamlit](#14.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Para um agente consultar sistemas externos, normalmente é preciso escrever uma integração para cada um. E a documentação técnica muda o tempo todo: um modelo treinado meses atrás não conhece os exemplos de código mais recentes.

### 0.2 Objetivo

- Registrar no **Microsoft Foundry** o servidor MCP público do **Microsoft Learn** (com governança e observabilidade do portal).
- Dar esse servidor como ferramenta (**`MCPTool`**) a um agente do **Foundry Agent Service**.
- Ver o agente **descobrir** as ferramentas do servidor e **escolher** a certa para cada pergunta.
- Conferir as chamadas e os tokens no **tracing**.

### 0.3 O que é MCP?

O **Model Context Protocol** é um padrão aberto para conectar modelos de IA a fontes de dados e ferramentas. Um **servidor MCP** anuncia uma lista de ferramentas (como os endpoints de uma API REST) e qualquer agente compatível pode listá-las e chamá-las.

| Ferramenta do servidor Microsoft Learn | O que faz |
| --- | --- |
| `microsoft_docs_search` | Busca na documentação oficial (Azure, .NET, Microsoft 365…) os trechos mais relevantes. |
| `microsoft_code_sample_search` | Busca **exemplos de código** oficiais (pode filtrar por linguagem). |
| `microsoft_docs_fetch` | Lê uma página completa da documentação. |

### 0.4 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Registra o servidor MCP como **conexão** (área *Tools*), com governança e observabilidade. |
| **Foundry Agent Service** | Hospeda o `MCP-Agent` e faz as chamadas ao servidor MCP. |
| **`MCPTool`** | Ferramenta do agente que aponta para o servidor e para a conexão do projeto. |
| **Connections** (`project_client.connections`) | Lista as conexões do projeto para obter o `connection.id`. |
| **Azure OpenAI** (deployment GPT-4) | Escolhe a ferramenta MCP e escreve a resposta. |
| **Microsoft Entra ID** | Autenticação sem chave (`DefaultAzureCredential`). |
| **Microsoft Learn MCP Server** | Servidor público e gratuito: `https://learn.microsoft.com/api/mcp`. |

### 0.5 Como funciona (visão simples)

![Fluxo do agente com MCP](fluxo_mcp.svg)

### 0.6 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

> 💡 **Para um leigo:** é como dar ao assistente um "crachá" de acesso à biblioteca da Microsoft. Ele pergunta ao bibliotecário (servidor MCP) quais serviços existem, escolhe o certo ("buscar exemplos de código") e volta com a resposta e os links.

---
## 1. Pré-requisito: registrar o servidor MCP no portal

Todo servidor MCP (seu ou de um fornecedor verificado, como a Microsoft) precisa ser **registrado no projeto** antes do uso, para ter governança e observabilidade:

1. Portal do Foundry → **Build** (Construir) → **Tools** (Ferramentas) → **Connect tool**.
2. Aba **Custom** → **Model Context Protocol (MCP)** → **Create**.
3. Preencher:

| Campo | Valor |
| --- | --- |
| Nome | ex.: `ms-learn-mcp-server` (é este nome que vai no `.env`) |
| Remote MCP Server endpoint | `https://learn.microsoft.com/api/mcp` |
| Authentication | **Unauthenticated** (servidor público). Para servidores internos há opções com **chave**, **Microsoft Entra ID** ou **OAuth identity passthrough**. |

4. **Connect**. O servidor passa a aparecer na lista de ferramentas do projeto.

> ⚠️ O valor de `MCP_SERVER_NAME` no `.env` precisa ser **exatamente** o nome dado à conexão no portal; caso contrário, o ID da conexão não é encontrado.

---
## 2. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |


> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 3. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |
| `MCP_SERVER_NAME` | Nome da conexão MCP criada no portal. | Portal do Foundry → **Tools** → nome da conexão (seção 1). |

```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>
MCP_SERVER_NAME=<mcp-server-name>
```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

Classes importadas: `PromptAgentDefinition` (o agente), `MCPTool` (a ferramenta MCP) e `Tool` (tipo base das ferramentas).

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# MCP_SERVER_NAME = nome da conexão MCP registrada no portal
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

---
## 4. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 5. Cliente OpenAI do projeto

O agente roda sobre um modelo **GPT** (Azure OpenAI). Para conversar com ele usamos a **Responses API** do SDK `openai`, e o próprio projeto entrega esse cliente já configurado com o endpoint e a autenticação do Foundry: `project_client.get_openai_client()`.

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
openai_client = project_client.get_openai_client()

---
## 6. Obtendo o ID da conexão MCP

Cada ferramenta registrada no portal vira uma **conexão** do projeto, com um **ID**. É por esse ID que o agente recebe permissão para usar a ferramenta.

O código percorre `project_client.connections.list()` e guarda o `id` da conexão cujo nome é igual a `MCP_SERVER_NAME`. O ID tem este formato:

```
/subscriptions/<id-da-assinatura>/resourceGroups/<grupo-de-recursos>/providers/Microsoft.CognitiveServices/accounts/<recurso-foundry>/projects/<projeto>/connections/<nome-da-conexão>
```

> 💡 O mesmo padrão (listar conexões e filtrar pelo nome) serve para qualquer outra ferramenta conectada ao projeto, como a automação de navegador do Lab 19.

In [ ]:
# Procurando, entre as conexões do projeto, a que tem o nome do servidor MCP
# e guardando o seu ID (é ele que autoriza o agente a usar a ferramenta)
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

---
## 7. Montando a ferramenta MCP

| Parâmetro | Valor | Significado |
| --- | --- | --- |
| `server_label` | `microsoft_learn_server` | Apelido do servidor (identifica a ferramenta no agente). |
| `server_url` | `https://learn.microsoft.com/api/mcp` | Endereço remoto do servidor MCP. |
| `require_approval` | `"never"` | O agente chama o servidor **sem pedir aprovação**. O padrão é pedir aprovação a cada chamada, o que exigiria tratar esse pedido no código (um botão "aprovar" ou um callback). |
| `project_connection_id` | `connection_id` | A conexão registrada no portal (seção 6). |

> ⚠️ `require_approval="never"` é prático em laboratório e para servidores confiáveis e somente leitura. Para ferramentas que **alteram dados**, manter a aprovação humana é uma boa prática de segurança.

In [ ]:
# Montando a ferramenta MCP: apelido, URL do servidor, sem aprovação manual
# e o ID da conexão registrada no portal
tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

---
## 8. Criando o agente MCP

| Campo | Valor |
| --- | --- |
| `agent_name` | `MCP-Agent` |
| `instructions` | *"Você é um assistente inteligente que pode interagir com o servidor MCP do Microsoft Learn para fornecer aos usuários recursos de aprendizado relevantes e informações sobre tecnologias Microsoft."* |
| `tools` | `[tool]` (o `MCPTool` da seção 7) |

In [ ]:
# Criando o agente com o servidor MCP do Microsoft Learn como ferramenta
agent = project_client.agents.create_version(
    agent_name="MCP-Agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
        tools=[tool],
    )
)

print(f"Created MCP Agent with ID: {agent.id}")

---
## 9. Criando a conversa

A **conversation** é um objeto guardado na Azure que **armazena o histórico** entre o usuário e o agente, identificado pelo `conversation.id`. Todas as mensagens enviadas com o mesmo id fazem parte do mesmo diálogo, então o agente "lembra" o que já foi dito.

> 💡 No portal, o mesmo `conversation.id` aparece em **Traces**, com cada chamada de ferramenta e o consumo de tokens.

In [ ]:
# Criando a conversa que guarda o histórico (memória) entre usuário e agente
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 10. Perguntando ao agente

Pergunta: *"Você pode me ajudar com os exemplos de código mais recentes do SDK do AI Foundry?"*

Também vale testar a pergunta do comentário: *"Encontre trilhas de aprendizado sobre serviços de Azure AI para criar aplicações inteligentes."*

In [ ]:
# Pergunta do usuário (o comentário abaixo traz uma segunda opção para testar)
user_query = "Can you pls help me with the latest ai foundry SDK code samples?" 

# Can also try this query: "Find me learning paths on Azure AI services for building intelligent applications."

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo. |
| `extra_body` → `agent` | `{"name": "MCP-Agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem versão informada, usa **a mais recente**. |
| `input` | pergunta do usuário | O texto enviado ao agente. |

A resposta final em texto fica em **`response.output_text`**. Repare que a pergunta **não manda** o agente usar a ferramenta: é o próprio agente que analisa a intenção e decide.

In [ ]:
# Enviando a pergunta: o agente lista as ferramentas do servidor MCP, escolhe e chama a certa
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": "MCP-Agent",
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Agent Response: {response.output_text}")

### 10.1 O que aconteceu na demonstração da aula

| Pergunta | Resposta do agente |
| --- | --- |
| Exemplos de código do SDK do AI Foundry | Exemplos para começar a criar agentes com o Foundry e **links oficiais** do learn.microsoft.com (incluindo um sobre *fine-tuning*). |
| Trilhas de aprendizado de Azure AI | *Get started with Azure AI*, *Develop generative AI apps in Azure* (~8 h), *Develop natural language solutions*, *Develop AI agents on Azure* (9 módulos)… |

✅ Uma "busca turbinada" no learn.microsoft.com, em linguagem natural e com links.

---
## 11. Verificando no portal do Foundry

Em **Traces**, abrindo o `conversation.id`, aparece a sequência completa:

| Etapa | O que aconteceu |
| --- | --- |
| `mcp_list_tools` | O agente perguntou ao servidor quais ferramentas existem e recebeu `microsoft_docs_search`, `microsoft_code_sample_search`… com suas descrições. |
| 1ª pergunta → `mcp_call` | `microsoft_code_sample_search` com os argumentos `query = "AI Foundry SDK"` e `language = "python"`; também chamou `microsoft_docs_search` para complementar. |
| 2ª pergunta → `mcp_call` | `microsoft_docs_search` com a consulta "learning paths Azure AI intelligent applications". |

| Execução (mesma conversa) | Tokens |
| --- | --- |
| 1ª pergunta | 8.382 |
| 2ª pergunta | 17.179 |

> 💡 A segunda execução carrega o histórico da primeira **mais** o conteúdo retornado pelo Microsoft Learn, por isso os tokens quase dobram.

---
## 12. Resumo dos aprendizados

- **MCP** é um padrão aberto: o servidor anuncia ferramentas e o agente as descobre (`mcp_list_tools`) e chama (`mcp_call`).
- Servidores MCP são **registrados como conexões** no portal do Foundry (governança e observabilidade).
- O código busca o **`connection.id`** com `connections.list()` e o passa ao `MCPTool`.
- `require_approval="never"` deixa o agente autônomo. O padrão (aprovação) é mais seguro para ferramentas que alteram dados.
- O **tracing** mostra cada ferramenta MCP usada, com argumentos e custo em tokens.

---
## 13. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Servidor MCP do Microsoft Learn | Gratuito e público. A conexão no portal pode ser removida em **Tools** quando não for mais usada. |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 14. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 4) e `get_openai_client()` (seção 5) |
| `nova_conversa() / perguntar()` | `conversations.create` (seção 9) e `responses.create` (seção 10) |
| `chamadas_mcp()` | lê as etapas `mcp_list_tools` e `mcp_call` da resposta: as mesmas vistas no tracing (seção 11) |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` e o histórico ficam no `st.session_state`.
- O agente **não é recriado** a cada mensagem: o app usa o agente já criado neste notebook (pelo nome, sempre na versão mais recente).
- Abaixo de cada resposta, um bloco expansível mostra **quais ferramentas do servidor MCP** o agente listou e chamou, com os argumentos.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
MCP_SERVER_NAME = os.getenv("MCP_SERVER_NAME")
AGENT_NAME = "MCP-Agent"  # criado na seção 8 do notebook

st.set_page_config(page_title="Microsoft Learn Agent · MCP", page_icon="📚", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Conversa e pergunta
# ------------------------------------------------------------------
def nova_conversa():
    st.session_state.conversation_id = criar_clientes().conversations.create().id
    st.session_state.messages = []


def perguntar(texto):
    return criar_clientes().responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )


# ------------------------------------------------------------------
# 4. O que o agente fez no servidor MCP (o mesmo que aparece no tracing)
# ------------------------------------------------------------------
def chamadas_mcp(response):
    chamadas = []
    for item in response.output:
        if item.type == "mcp_list_tools":
            nomes = ", ".join(tool.name for tool in getattr(item, "tools", []) or [])
            chamadas.append(f"📋 listou as ferramentas: {nomes}")
        elif item.type == "mcp_call":
            chamadas.append(f"🔧 chamou **{item.name}** com `{item.arguments}`")
    return chamadas


def mostrar_chamadas(chamadas):
    if chamadas:
        with st.expander(f"🛰️ Chamadas ao servidor MCP ({len(chamadas)})"):
            for chamada in chamadas:
                st.markdown(f"- {chamada}")


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("📚 Agente do Microsoft Learn")
st.caption("Foundry Agent Service + servidor MCP do Microsoft Learn: documentação, exemplos de código e trilhas de estudo em linguagem natural.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("🛰️ Servidor MCP")
    st.caption(f"Conexão: `{MCP_SERVER_NAME}`  \nURL: `https://learn.microsoft.com/api/mcp`  \nAgente: `{AGENT_NAME}`")
    st.caption(f"Conversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()
    st.divider()
    st.markdown("**Sugestões**")
    st.caption("• Exemplos de código do SDK do Microsoft Foundry em Python  \n• Trilhas de aprendizado de Azure AI para apps inteligentes")

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        mostrar_chamadas(message.get("chamadas"))

if prompt := st.chat_input("Pergunte sobre qualquer tecnologia Microsoft"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Consultando o Microsoft Learn..."):
            try:
                response = perguntar(prompt)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                chamadas = chamadas_mcp(response)
                st.markdown(response.output_text)
                mostrar_chamadas(chamadas)
                st.session_state.messages.append({"role": "assistant", "content": response.output_text, "chamadas": chamadas})
